In [1]:
import os

def show_inputs(depth=4, max_files_per_dir=6):
    """Print what is attached under /kaggle/input - check it before trusting the rest."""
    root0 = os.environ.get("SAT_INPUT", "/kaggle/input")
    for root, dirs, files in os.walk(root0):
        dirs[:] = sorted(d for d in dirs if not d.startswith("."))
        level = root.replace(root0, "").count(os.sep)
        if level > depth:
            dirs[:] = []
            continue
        print("  " * level + root + "/")
        for fn in sorted(files)[:max_files_per_dir]:
            print("  " * level + "  " + fn)
        if len(files) > max_files_per_dir:
            print("  " * level + f"  ... +{len(files) - max_files_per_dir} more files")

show_inputs()

/kaggle/input/
  /kaggle/input/datasets/
    /kaggle/input/datasets/rifat4018/
      /kaggle/input/datasets/rifat4018/sat-ext/
        /kaggle/input/datasets/rifat4018/sat-ext/sat_ext/
          README.md
          audit.py
          config.py
          dataset.py
          infer.py
          metrics.py
          ... +7 more files
      /kaggle/input/datasets/rifat4018/sat-splits-sky/
        /kaggle/input/datasets/rifat4018/sat-splits-sky/audit/
        /kaggle/input/datasets/rifat4018/sat-splits-sky/splits/
      /kaggle/input/datasets/rifat4018/skysealand-v14/
        README.dataset.txt
        README.roboflow.txt
        /kaggle/input/datasets/rifat4018/skysealand-v14/test/
          Screenshot-2025-06-24-155441_png.rf.2f4261409df2feaaf1bed35c5f8ab44d.jpg
          Screenshot-2025-06-24-160143_png.rf.5ca695e301401bcb5aa4bfa8e3dde276.jpg
          Screenshot-2025-06-24-162141_png.rf.0ebb49d1838a17cc7da44c7a37575cac.jpg
          Screenshot-2025-06-24-162408_png.rf.8222548ffbf5e5fa97

In [2]:
import os, sys, warnings
warnings.filterwarnings("ignore")
INPUT_ROOT = os.environ.get("SAT_INPUT", "/kaggle/input")

def _find_code_root():
    for root, dirs, files in os.walk(INPUT_ROOT):
        if {"config.py", "audit.py", "metrics.py", "infer.py"} <= set(files):
            return root
    return None

CODE_ROOT = os.environ.get("SAT_CODE") or _find_code_root()
assert CODE_ROOT, "sat_ext code folder not found - Add Input -> your 'sat-ext' code Dataset"
sys.path.insert(0, CODE_ROOT)
print("CODE_ROOT:", CODE_ROOT)

import numpy as np, pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 60); pd.set_option("display.max_colwidth", 70)
import config as cfg, utils
utils.ensure_dirs(); utils.set_seed(cfg.SEED)
print("work dir:", cfg.WORK_DIR, "| smoke test:", cfg.SMOKE)

CODE_ROOT: /kaggle/input/datasets/rifat4018/sat-ext/sat_ext
work dir: /kaggle/working | smoke test: False


In [3]:
KEY = "G"            # preset - do not change
PILOT = False         # True = 1-epoch YOLO11n check of the whole GPU pipeline (tagged __pilot, ignored by the tables)
import plans, train
PLAN = plans.PILOT if PILOT else plans.PLANS[KEY]
EPOCHS = 1 if (PILOT or cfg.SMOKE) else None
TAG = "pilot" if PILOT else ""
if PILOT:
    print("PILOT: grouped / YOLO11n / seed 42, 1 epoch")
else:
    plans.show(KEY)
print("epochs:", EPOCHS or cfg.TRAIN_PARAMS["epochs"])

02G-train-skysealand-5models-s42: 5 runs, ~5.1 GPU h (planning estimate)
  split=sky_grouped  model=YOLO11n    seed=42
  split=sky_grouped  model=YOLO11s    seed=42
  split=sky_grouped  model=YOLO11m    seed=42
  split=sky_grouped  model=YOLOv8m    seed=42
  split=sky_grouped  model=RT-DETR-L  seed=42
epochs: 50


In [4]:
utils.require_gpu()
assert utils.install_detection_stack(), "pip install failed - is Internet ON? (Settings -> Internet)"
print(utils.versions())

GPU: {'device': 'cuda:0', 'gpu': 'Tesla T4', 'n_gpu': 2, 'torch': '2.11.0+cu128'}
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
{'ultralytics': '8.4.37', 'sahi': '0.12.8', 'torch': '2.11.0+cu128', 'pycocotools': 'installed', 'numpy': '2.1.3', 'cv2': '4.14.0'}


In [5]:
done = train.run_plan(PLAN, epochs=EPOCHS, tag=TAG)

materialized 'sky_grouped' (0b03f1f732a3) -> /kaggle/working/data_sky_grouped  images {'train': 922, 'val': 193, 'test': 192}
New https://pypi.org/project/ultralytics/8.4.183 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.37 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data_sky_grouped/data.yaml, degrees=45.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr

2026-10-20 07:43:35,854 - sahi - WARNING - ignoring invalid prediction with bbox: [1.1963410377502441, 0, 109.22480773925781, 0] (ultralytics.py:278)


  sky_grouped__YOLO11n__s42    sahi480   AP50=0.7189 AP=0.3158 AR50_micro=0.319 | conf>=0.25: dets=7000 TP=2403 FP=4597 R=0.860 | 183.7 ms/img


2026-10-20 07:44:53,318 - sahi - WARNING - ignoring invalid prediction with bbox: [1.1963410377502441, 0, 109.22480773925781, 0] (ultralytics.py:278)


  sky_grouped__YOLO11n__s42    sahi320   AP50=0.6086 AP=0.2709 AR50_micro=0.396 | conf>=0.25: dets=10675 TP=2381 FP=8294 R=0.852 | 343.2 ms/img


...

  sky_grouped__YOLOv8m__s42    sahi480   AP50=0.6486 AP=0.2459 AR50_micro=0.222 | conf>=0.25: dets=7028 TP=2258 FP=4770 R=0.808 | 303.0 ms/img


2026-10-20 09:32:10,137 - sahi - WARNING - ignoring invalid prediction with bbox: [987.703857421875, 0, 1051.443603515625, 0] (ultralytics.py:278)


  sky_grouped__YOLOv8m__s42    sahi320   AP50=0.6014 AP=0.2569 AR50_micro=0.222 | conf>=0.25: dets=10594 TP=2310 FP=8284 R=0.826 | 629.7 ms/img


2026-10-20 09:37:02,708 - sahi - WARNING - ignoring invalid prediction with bbox: [987.703857421875, 0, 1051.443603515625, 0] (ultralytics.py:278)


  sky_grouped__YOLOv8m__s42    sahi160   AP50=0.4915 AP=0.2203 AR50_micro=0.174 | conf>=0.25: dets=34533 TP=2308 FP=32225 R=0.826 | 2140.1 ms/img
New https://pypi.org/project/ultralytics/8.4.183 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.37 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data_sky_grouped/data.yaml, degrees=45.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, 

In [6]:
import shutil, glob
for rid, status in done:
    ev = os.path.join(cfg.RESULTS_DIR, rid, "eval_test.json")
    if os.path.isfile(ev):
        d = utils.load_json(ev)
        u = d["ultralytics_test"]
        print(f"{rid:<30} test mAP50={u['mAP50']:.4f} mAP50-95={u['mAP50_95']:.4f} | " +
              " ".join(f"{m}:AP={v['coco'].get('AP_all', float('nan')):.3f}" for m, v in d["modes"].items()))
z = shutil.make_archive(os.path.join(cfg.WORK_DIR, f"results_{KEY}"), "zip", cfg.RESULTS_DIR)
print("\nsmall results zip (no weights):", z, f"{os.path.getsize(z) / 1e6:.1f} MB")
for f in glob.glob(os.path.join(cfg.WORK_DIR, "runs")):
    shutil.rmtree(f, ignore_errors=True)       # Ultralytics run folders are duplicated in results/ + weights/

sky_grouped__YOLO11n__s42      test mAP50=0.7832 mAP50-95=0.4075 | standard:AP=0.346 sahi480:AP=0.316 sahi320:AP=0.271 sahi160:AP=0.141
sky_grouped__YOLO11s__s42      test mAP50=0.8278 mAP50-95=0.4561 | standard:AP=0.410 sahi480:AP=0.385 sahi320:AP=0.330 sahi160:AP=0.239
sky_grouped__YOLO11m__s42      test mAP50=0.8214 mAP50-95=0.4323 | standard:AP=0.274 sahi480:AP=0.222 sahi320:AP=0.237 sahi160:AP=0.226
sky_grouped__YOLOv8m__s42      test mAP50=0.8475 mAP50-95=0.4718 | standard:AP=0.374 sahi480:AP=0.246 sahi320:AP=0.257 sahi160:AP=0.220
sky_grouped__RT-DETR-L__s42    test mAP50=0.6470 mAP50-95=0.3208 | standard:AP=0.321 sahi480:AP=0.297 sahi320:AP=0.310 sahi160:AP=0.291

small results zip (no weights): /kaggle/working/results_G.zip 259.3 MB
